# Silver Date Dimension Table

Generates `caresync_databricks_workspace.silver.dim_date` covering 1970–2030 with calendar columns, fiscal year (January start), US holiday flags, and business day indicators.

In [0]:
%pip install holidays --quiet

In [0]:
import holidays as pyholidays
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DateType, StringType

# --- Configuration ---
START_DATE = date(1970, 1, 1)
END_DATE = date(2030, 12, 31)
TARGET_TABLE = "caresync_databricks_workspace.silver.dim_date"

# --- Generate US holiday reference ---
us_holidays = pyholidays.US(years=range(1970, 2031))
holiday_rows = [(d, name) for d, name in us_holidays.items()]
holiday_schema = StructType([
    StructField("holiday_date", DateType(), False),
    StructField("holiday_name", StringType(), True),
])
holidays_df = spark.createDataFrame(holiday_rows, schema=holiday_schema)

# --- Generate date spine (1970-01-01 to 2030-12-31) ---
num_days = (END_DATE - START_DATE).days + 1
date_spine = (
    spark.range(num_days)
    .withColumn("full_date", F.date_add(F.lit(START_DATE), F.col("id").cast("int")))
    .drop("id")
)

# --- Build dimension columns ---
dim_date = (
    date_spine
    .join(holidays_df, date_spine["full_date"] == holidays_df["holiday_date"], "left")
    # Surrogate key
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    # Calendar grain
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("month_abbr", F.date_format("full_date", "MMM"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))       # 1=Sun … 7=Sat
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("day_abbr", F.date_format("full_date", "EEE"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("day_of_year", F.dayofyear("full_date"))
    # Fiscal year (January start = calendar year)
    .withColumn("fiscal_year", F.year("full_date"))
    .withColumn("fiscal_quarter", F.quarter("full_date"))
    # Convenience groupings
    .withColumn("year_month", F.date_format("full_date", "yyyy-MM"))
    .withColumn("year_quarter",
        F.concat(F.col("year").cast("string"), F.lit("-Q"), F.col("quarter").cast("string"))
    )
    # Flags
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
    .withColumn("is_month_start", F.dayofmonth("full_date") == 1)
    .withColumn("is_month_end", F.col("full_date") == F.last_day("full_date"))
    .withColumn("is_us_holiday", F.col("holiday_name").isNotNull())
    .withColumn("us_holiday_name", F.col("holiday_name"))
    .withColumn("is_business_day",
        ~(F.dayofweek("full_date").isin(1, 7) | F.col("holiday_name").isNotNull())
    )
    .drop("holiday_date", "holiday_name")
    .orderBy("full_date")
)

# --- Write to Delta ---
dim_date.write.mode("overwrite").saveAsTable(TARGET_TABLE)

print(f"\u2713 {TARGET_TABLE} created successfully")
display(spark.table(TARGET_TABLE))